# Ensemble Method — SNAPSHOT ENSEMBLE
Trains a single XGBoost model through multiple learning-rate cycles.
A checkpoint (snapshot) is saved at the trough of each cycle and the snapshots are blended.
Delivers diversity at the cost of one training run.
Optuna tunes the base LR, cycle length, number of cycles, and blend sharpening alpha.

In [1]:
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, matthews_corrcoef
import xgboost as xgb
import warnings

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 1. Load Data

In [2]:
BASE = r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\ranked rf"

def load_numeric_features(filepath):
    df = pd.read_csv(filepath, low_memory=False)
    return df.select_dtypes(include=[np.number]).values.astype(float)

X_train = load_numeric_features(f"{BASE}\\X_train_rF_DEG_train.csv")
X_test  = load_numeric_features(f"{BASE}\\X_test_rF_DEG_train.csv")
y_train = pd.read_csv(f"{BASE}\\y_train_rF_DEG_train.csv", low_memory=False)["cancer_type"].values.astype(int)
y_test  = pd.read_csv(f"{BASE}\\y_test_rF_DEG_train.csv",  low_memory=False)["cancer_type"].values.astype(int)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_train: (1615, 537) | X_test: (404, 537)


## 2. Snapshot Training Helper
We simulate a cyclic LR schedule by training `n_cycles` separate XGBoost models,
each warm-started from the previous one but at a decaying base learning rate.
Each "cycle end" model is one snapshot.

In [3]:
def cosine_annealing_lrs(base_lr, n_cycles, cycle_length):
    """
    Returns list of (start_lr, end_lr) per cycle.
    LR follows a cosine curve from base_lr → near 0 within each cycle,
    and the base_lr decays by half each cycle.
    """
    lrs = []
    for c in range(n_cycles):
        cycle_base = base_lr / (2 ** c)      # halve each cycle
        cycle_min  = cycle_base * 0.01       # trough ≈ 1% of base
        lrs.append((cycle_base, cycle_min))
    return lrs


def train_snapshots(X, y, params, n_cycles, cycle_length):
    """
    Trains n_cycles XGBoost models in sequence (simulated snapshot ensemble).
    Returns list of fitted models (one per snapshot).
    """
    lrs   = cosine_annealing_lrs(params["base_lr"], n_cycles, cycle_length)
    snapshots = []

    for c, (lr_start, lr_end) in enumerate(lrs):
        # Use the average LR across the cycle as a proxy
        cycle_lr = (lr_start + lr_end) / 2.0

        model = xgb.XGBClassifier(
            n_estimators     = cycle_length,
            learning_rate    = cycle_lr,
            max_depth        = params["max_depth"],
            subsample        = params["subsample"],
            colsample_bytree = params["colsample_bytree"],
            min_child_weight = params["min_child_weight"],
            reg_alpha        = params["reg_alpha"],
            reg_lambda       = params["reg_lambda"],
            eval_metric="logloss",
            random_state=42 + c,   # different seed per snapshot for diversity
            n_jobs=-1,
        )
        model.fit(X, y)
        snapshots.append(model)

    return snapshots

## 3. Optuna Objective

In [4]:
def objective(trial):
    # ── Snapshot / cycle parameters ────────────────────────────────────────
    n_cycles     = trial.suggest_int(  "n_cycles",     3, 8)
    cycle_length = trial.suggest_int(  "cycle_length", 50, 200, step=25)
    alpha        = trial.suggest_float("alpha",        1.0, 15.0)  # blend sharpening

    # ── XGBoost base parameters ────────────────────────────────────────────
    params = {
        "base_lr"        : trial.suggest_float("base_lr",         0.05, 0.3,  log=True),
        "max_depth"      : trial.suggest_int(  "max_depth",       3, 9),
        "subsample"      : trial.suggest_float("subsample",       0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int(  "min_child_weight", 1, 10),
        "reg_alpha"      : trial.suggest_float("reg_alpha",       1e-4, 10.0, log=True),
        "reg_lambda"     : trial.suggest_float("reg_lambda",      1e-4, 10.0, log=True),
    }

    # ── Cross-validate the full snapshot blend ─────────────────────────────
    oof_blend = np.zeros(X_train.shape[0])

    for fold, (tr_idx, val_idx) in enumerate(CV.split(X_train, y_train)):
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        snapshots = train_snapshots(X_tr, y_tr, params, n_cycles, cycle_length)

        # AUROC-weighted BMA across snapshots
        snap_probas  = [s.predict_proba(X_val)[:, 1] for s in snapshots]
        snap_aurocs  = [roc_auc_score(y_val, p) for p in snap_probas]
        raw_weights  = [np.exp(alpha * a) for a in snap_aurocs]
        total_w      = sum(raw_weights)
        oof_blend[val_idx] = sum((w / total_w) * p
                                  for w, p in zip(raw_weights, snap_probas))

    best_mcc = -1.0
    for t in np.linspace(0.2, 0.8, 61):
        mcc = matthews_corrcoef(y_train, (oof_blend >= t).astype(int))
        if mcc > best_mcc:
            best_mcc = mcc

    return best_mcc

## 4. Run Optimization

In [5]:
print("Starting Snapshot Ensemble Optimization...")

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\n🏆 Best OOF MCC: {study.best_value:.4f}")
print("Best Parameters:")
for k, v in sorted(study.best_params.items()):
    print(f"  {k:25s} = {v}")

Starting Snapshot Ensemble Optimization...


  0%|          | 0/50 [00:00<?, ?it/s]


🏆 Best OOF MCC: 0.6325
Best Parameters:
  alpha                     = 10.72655517742869
  base_lr                   = 0.17247298025406013
  colsample_bytree          = 0.9672472583004309
  cycle_length              = 150
  max_depth                 = 8
  min_child_weight          = 4
  n_cycles                  = 5
  reg_alpha                 = 0.002423275827338096
  reg_lambda                = 0.23608065526609617
  subsample                 = 0.5167929424672362


## 5. Final Training & Evaluation

In [6]:
best = study.best_params

final_params = {
    "base_lr"         : best["base_lr"],
    "max_depth"       : best["max_depth"],
    "subsample"       : best["subsample"],
    "colsample_bytree": best["colsample_bytree"],
    "min_child_weight": best["min_child_weight"],
    "reg_alpha"       : best["reg_alpha"],
    "reg_lambda"      : best["reg_lambda"],
}

print(f"Training final snapshot ensemble ({best['n_cycles']} snapshots × {best['cycle_length']} trees)...")
final_snapshots = train_snapshots(
    X_train, y_train, final_params, best["n_cycles"], best["cycle_length"]
)

# ── Generate weighted blend on test set ───────────────────────────────────
snap_test_probas = [s.predict_proba(X_test)[:, 1]  for s in final_snapshots]
snap_oof_probas  = [s.predict_proba(X_train)[:, 1] for s in final_snapshots]

snap_aurocs  = [roc_auc_score(y_train, p) for p in snap_oof_probas]
raw_weights  = [np.exp(best["alpha"] * a) for a in snap_aurocs]
total_w      = sum(raw_weights)
norm_weights = [w / total_w for w in raw_weights]

final_test_blend = sum(w * p for w, p in zip(norm_weights, snap_test_probas))
final_oof_blend  = sum(w * p for w, p in zip(norm_weights, snap_oof_probas))

# ── Threshold calibration ─────────────────────────────────────────────────
best_thr, best_mcc = 0.5, -1.0
for t in np.linspace(0.1, 0.9, 81):
    mcc = matthews_corrcoef(y_train, (final_oof_blend >= t).astype(int))
    if mcc > best_mcc:
        best_mcc, best_thr = mcc, t

y_pred = (final_test_blend >= best_thr).astype(int)

print("\n" + "="*55)
print("  SNAPSHOT ENSEMBLE — FINAL RESULTS")
print("="*55)
print(f"  Snapshots            : {best['n_cycles']}")
print(f"  Trees per snapshot   : {best['cycle_length']}")
print(f"  Snapshot Weights     :")
for i, (w, a) in enumerate(zip(norm_weights, snap_aurocs)):
    print(f"    Snapshot {i+1}: weight={w:.4f}  OOF-AUROC={a:.4f}")
print(f"\n  Decision Threshold   : {best_thr:.2f}")
print(f"  Test AUROC           : {roc_auc_score(y_test, final_test_blend):.4f}")
print(f"  Test MCC             : {matthews_corrcoef(y_test, y_pred):.4f}")
print("="*55)

Training final snapshot ensemble (5 snapshots × 150 trees)...

  SNAPSHOT ENSEMBLE — FINAL RESULTS
  Snapshots            : 5
  Trees per snapshot   : 150
  Snapshot Weights     :
    Snapshot 1: weight=0.2075  OOF-AUROC=1.0000
    Snapshot 2: weight=0.2072  OOF-AUROC=0.9999
    Snapshot 3: weight=0.2053  OOF-AUROC=0.9990
    Snapshot 4: weight=0.1958  OOF-AUROC=0.9946
    Snapshot 5: weight=0.1842  OOF-AUROC=0.9889

  Decision Threshold   : 0.61
  Test AUROC           : 0.9139
  Test MCC             : 0.5747


In [7]:
from sklearn.metrics import classification_report
# Reslut for testing set
from sklearn.metrics import roc_curve, auc


# print("Training time:", training_time, "seconds")
print(classification_report( y_test,y_pred,digits=6))


              precision    recall  f1-score   support

           0   0.725806  0.576923  0.642857        78
           1   0.903509  0.947853  0.925150       326

    accuracy                       0.876238       404
   macro avg   0.814658  0.762388  0.784003       404
weighted avg   0.869200  0.876238  0.870648       404

